install and import libraries

In [1]:
#pip install pandas

In [2]:
import pandas as pd
import glob
import sqlite3

biometric data

In [3]:
bio_files = glob.glob("../data/raw/biometric/*.csv")
biometric_df = pd.concat(
    (pd.read_csv(f) for f in bio_files),
    ignore_index=True
)
print("Biometric rows:", biometric_df.shape)

Biometric rows: (1861108, 6)


In [4]:
biometric_df.head()

,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu and Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [5]:
biometric_df.isnull().sum()

date            0
state           0
district        0
pincode         0
bio_age_5_17    0
bio_age_17_     0
dtype: int64

Demographic data

In [6]:
demo_files = glob.glob("../data/raw/demographic/*.csv")
demographic_df = pd.concat(
    (pd.read_csv(f) for f in demo_files),
    ignore_index=True
)
print("Demographic rows:", demographic_df.shape)

Demographic rows: (2071700, 6)


In [7]:
demographic_df.head()

,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,Uttar Pradesh,Gorakhpur,273213,49,529
1,01-03-2025,Andhra Pradesh,Chittoor,517132,22,375
2,01-03-2025,Gujarat,Rajkot,360006,65,765
3,01-03-2025,Andhra Pradesh,Srikakulam,532484,24,314
4,01-03-2025,Rajasthan,Udaipur,313801,45,785


In [8]:
demographic_df.isnull().sum()

date             0
state            0
district         0
pincode          0
demo_age_5_17    0
demo_age_17_     0
dtype: int64

Enrolment data

In [9]:
enr_files = glob.glob("../data/raw/enrolment/*.csv")
enrolment_df = pd.concat(
    (pd.read_csv(f) for f in enr_files),
    ignore_index=True
)
print("Enrolment rows:", enrolment_df.shape)

Enrolment rows: (1006029, 7)


In [10]:
enrolment_df.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,Meghalaya,East Khasi Hills,793121,11,61,37
1,09-03-2025,Karnataka,Bengaluru Urban,560043,14,33,39
2,09-03-2025,Uttar Pradesh,Kanpur Nagar,208001,29,82,12
3,09-03-2025,Uttar Pradesh,Aligarh,202133,62,29,15
4,09-03-2025,Karnataka,Bengaluru Urban,560016,14,16,21


In [11]:
enrolment_df.isnull().sum()

date              0
state             0
district          0
pincode           0
age_0_5           0
age_5_17          0
age_18_greater    0
dtype: int64

Standardizing the column names

Check for duplicates

In [12]:
print("Biometric duplicates:", biometric_df.duplicated().sum())
print("Demographic duplicates:", demographic_df.duplicated().sum())
print("Enrolment duplicates:", enrolment_df.duplicated().sum())

Biometric duplicates: 94896
Demographic duplicates: 473601
Enrolment duplicates: 22957


In [28]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

def standardize_states(df):
    """Standardize state names to official names (post-2020 UT merger)."""
    state_mapping = {
        # UT merger: map separate names to merged UT
        'dadra & nagar haveli': 'Dadra and Nagar Haveli and Daman and Diu',
        'dadra and nagar haveli': 'Dadra and Nagar Haveli and Daman and Diu',
        'daman & diu': 'Dadra and Nagar Haveli and Daman and Diu',
        'daman and diu': 'Dadra and Nagar Haveli and Daman and Diu',
        'dadra and nagar haveli and daman and diu': 'Dadra and Nagar Haveli and Daman and Diu',
        'the dadra and nagar haveli and daman and diu': 'Dadra and Nagar Haveli and Daman and Diu',

        # AND vs & inconsistencies
        'andaman and nicobar islands': 'Andaman & Nicobar Islands',
        'andaman & nicobar islands': 'Andaman & Nicobar Islands',
        'jammu and kashmir': 'Jammu & Kashmir',
        'jammu & kashmir': 'Jammu & Kashmir',

        # Case/spelling variations
        'odisha': 'Odisha',
        'orissa': 'Odisha',
        'west bengal': 'West Bengal',
        'west  bengal': 'West Bengal',
        'west bangal': 'West Bengal',
        'west bengli': 'West Bengal',
        'westbengal': 'West Bengal',
        'andhra pradesh': 'Andhra Pradesh',
        'tamilnadu': 'Tamil Nadu',
        'tamil nadu': 'Tamil Nadu',
        'uttaranchal': 'Uttarakhand',
        'pondicherry': 'Puducherry',
        'chhatisgarh': 'Chhattisgarh',

        # Invalid entries (cities, numbers) - remove
        '100000': None,
        'balanagar': None,
        'jaipur': None,
        'nagpur': None,
        'puttenahalli': None,
        'darbhanga': None,
        'madanapalle': None,
        'raja annamalai puram': None,
    }

    df['state'] = df['state'].str.strip().str.lower()
    df['state'] = df['state'].map(lambda x: state_mapping.get(x, x.title()))
    df = df[df['state'].notna()]
    return df

biometric_df = clean_columns(biometric_df)
demographic_df = clean_columns(demographic_df)
enrolment_df = clean_columns(enrolment_df)

# Apply state standardization
biometric_df = standardize_states(biometric_df)
demographic_df = standardize_states(demographic_df)
enrolment_df = standardize_states(enrolment_df)

print("✅ Column names standardized and state names unified (merged UT applied)!")
print(f"\nBiometric rows after cleaning: {len(biometric_df)}")
print(f"Demographic rows after cleaning: {len(demographic_df)}")
print(f"Enrolment rows after cleaning: {len(enrolment_df)}")

✅ Column names standardized and state names unified (merged UT applied)!

Biometric rows after cleaning: 1765691
Demographic rows after cleaning: 1597431
Enrolment rows after cleaning: 982584


In [25]:
# Remove duplicates - keep only first occurrence
print("🧹 REMOVING DUPLICATES...")
print(f"\nBefore duplicate removal:")
print(f"  • Biometric: {len(biometric_df)} rows")
print(f"  • Demographic: {len(demographic_df)} rows")
print(f"  • Enrolment: {len(enrolment_df)} rows")

biometric_df = biometric_df.drop_duplicates()
demographic_df = demographic_df.drop_duplicates()
enrolment_df = enrolment_df.drop_duplicates()

print(f"\nAfter duplicate removal:")
print(f"  • Biometric: {len(biometric_df)} rows (removed {1861108 - len(biometric_df)})")
print(f"  • Demographic: {len(demographic_df)} rows (removed {2071687 - len(demographic_df)})")
print(f"  • Enrolment: {len(enrolment_df)} rows (removed {1006007 - len(enrolment_df)})")
print(f"\n✅ Duplicates removed successfully!")

🧹 REMOVING DUPLICATES...

Before duplicate removal:
  • Biometric: 1861108 rows
  • Demographic: 2071687 rows
  • Enrolment: 1006007 rows

After duplicate removal:
  • Biometric: 1765691 rows (removed 95417)
  • Demographic: 1597431 rows (removed 474256)
  • Enrolment: 982584 rows (removed 23423)

✅ Duplicates removed successfully!


sql

In [30]:
conn = sqlite3.connect("../data/processed/uidai.db")

biometric_df.to_sql("biometric_updates", conn, if_exists="replace", index=False)
demographic_df.to_sql("demographic_updates", conn, if_exists="replace", index=False)
enrolment_df.to_sql("enrolments", conn, if_exists="replace", index=False)

982584

In [31]:
pd.read_sql(
    "SELECT state, COUNT(*) AS total FROM biometric_updates GROUP BY state",
    conn
)


,state,total
0,Andaman & Nicobar Islands,1687
1,Andhra Pradesh,160232
2,Arunachal Pradesh,3957
3,Assam,44418
4,Bihar,78078
5,Chandigarh,1576
6,Chhattisgarh,30053
7,Dadra and Nagar Haveli and Daman and Diu,1243
8,Delhi,8784
9,Goa,5105


In [16]:
pd.read_sql("""
SELECT state, COUNT(*) AS records
FROM enrolments
GROUP BY state
ORDER BY records DESC
LIMIT 5
""", conn)

,state,records
0,Uttar Pradesh,110369
1,Tamil Nadu,92552
2,Maharashtra,77191
3,West Bengal,76519
4,Karnataka,70198


Saving of cleaned data

In [32]:
biometric_df.to_csv("../data/processed/biometric_clean.csv", index=False)
demographic_df.to_csv("../data/processed/demographic_clean.csv", index=False)
enrolment_df.to_csv("../data/processed/enrolment_clean.csv", index=False)

## Data Quality Summary

In [33]:
print("=" * 80)
print("DATA LOADING & QUALITY SUMMARY")
print("=" * 80)

print("\n📊 BIOMETRIC DATA:")
print(f"  • Shape: {biometric_df.shape}")
print(f"  • Missing values: {biometric_df.isnull().sum().sum()}")
print(f"  • Date range: {biometric_df['date'].min()} to {biometric_df['date'].max()}")
print(f"  • Unique states: {biometric_df['state'].nunique()}")

print("\n📊 DEMOGRAPHIC DATA:")
print(f"  • Shape: {demographic_df.shape}")
print(f"  • Missing values: {demographic_df.isnull().sum().sum()}")
print(f"  • Date range: {demographic_df['date'].min()} to {demographic_df['date'].max()}")
print(f"  • Unique states: {demographic_df['state'].nunique()}")

print("\n📊 ENROLMENT DATA:")
print(f"  • Shape: {enrolment_df.shape}")
print(f"  • Missing values: {enrolment_df.isnull().sum().sum()}")
print(f"  • Date range: {enrolment_df['date'].min()} to {enrolment_df['date'].max()}")
print(f"  • Unique states: {enrolment_df['state'].nunique()}")

print("\n✅ DUPLICATE CHECK:")
print(f"  • Biometric duplicates: {biometric_df.duplicated().sum()}")
print(f"  • Demographic duplicates: {demographic_df.duplicated().sum()}")
print(f"  • Enrolment duplicates: {enrolment_df.duplicated().sum()}")

print("\n💾 DATA SAVED:")
print(f"  ✓ biometric_clean.csv")
print(f"  ✓ demographic_clean.csv")
print(f"  ✓ enrolment_clean.csv")
print(f"  ✓ uidai.db (SQLite)")

print("\n" + "=" * 80)
print("✨ Data loading complete and validated!")
print("=" * 80)

DATA LOADING & QUALITY SUMMARY

📊 BIOMETRIC DATA:
  • Shape: (1765691, 6)
  • Missing values: 0
  • Date range: 01-03-2025 to 31-10-2025
  • Unique states: 36

📊 DEMOGRAPHIC DATA:
  • Shape: (1597431, 6)
  • Missing values: 0
  • Date range: 01-03-2025 to 31-10-2025
  • Unique states: 36

📊 ENROLMENT DATA:
  • Shape: (982584, 7)
  • Missing values: 0
  • Date range: 01-04-2025 to 31-12-2025
  • Unique states: 36

✅ DUPLICATE CHECK:
  • Biometric duplicates: 10
  • Demographic duplicates: 45
  • Enrolment duplicates: 9

💾 DATA SAVED:
  ✓ biometric_clean.csv
  ✓ demographic_clean.csv
  ✓ enrolment_clean.csv
  ✓ uidai.db (SQLite)

✨ Data loading complete and validated!


In [29]:
print("=" * 100)
print("STATE NAME ANALYSIS - AFTER STANDARDIZATION")
print("=" * 100)

bio_states = sorted(biometric_df['state'].unique())
demo_states = sorted(demographic_df['state'].unique())
enr_states = sorted(enrolment_df['state'].unique())

print(f"\n📍 BIOMETRIC States ({len(bio_states)}):")
print(bio_states)

print(f"\n📍 DEMOGRAPHIC States ({len(demo_states)}):")
print(demo_states)

print(f"\n📍 ENROLMENT States ({len(enr_states)}):")
print(enr_states)

# Check for mismatches
all_states = set(bio_states) | set(demo_states) | set(enr_states)
print(f"\n✨ Total unique state names after standardization: {len(all_states)}")

bio_set = set(bio_states)
demo_set = set(demo_states)
enr_set = set(enr_states)

print(f"\n📊 STATE COVERAGE:")
print(f"  • Biometric only: {bio_set - demo_set - enr_set}")
print(f"  • Demographic only: {demo_set - bio_set - enr_set}")
print(f"  • Enrolment only: {enr_set - bio_set - demo_set}")
print(f"  • Common in all three datasets: {len(bio_set & demo_set & enr_set)} states")

print("\n" + "=" * 100)

STATE NAME ANALYSIS - AFTER STANDARDIZATION

📍 BIOMETRIC States (36):
['Andaman & Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']

📍 DEMOGRAPHIC States (36):
['Andaman & Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu & Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'O